As we will see, one of the hyperparameters that affects training the most is the learning rate. 
Let’s see that in practice in our toy example. Run the SGD example above with three other values 
for the learning rate: 1e1, 1e2, and 1e3, for just 10 training iterations. What happens with the loss 
for each of these learning rates? Does it decay faster, slower, or does it diverge (i.e., increase over 
the course of training)?
Deliverable: A one-to-two sentence response with the behaviors you observed.


In [1]:
from collections.abc import Callable, Iterable
from typing import Optional
import torch
import math


class SGD(torch.optim.Optimizer):
    def __init__(self, params, lr=1e-3):
        if lr < 0:
            raise ValueError(f"Invalid learning rate: {lr}")
        defaults = {"lr": lr}
        super().__init__(params, defaults)
    def step(self, closure: Optional[Callable] = None):
        loss = None if closure is None else closure()
        for group in self.param_groups:
            lr = group["lr"]  # Get the learning rate.
            for p in group["params"]:
                if p.grad is None:
                    continue
                state = self.state[p]  # Get state associated with p.
                t = state.get("t", 0)  # Get iteration number from the state, or 0.
                grad = p.grad.data  # Get the gradient of loss with respect to p.
                p.data -= lr / math.sqrt(t + 1) * grad  # Update weight tensor in-place.
                state["t"] = t + 1  # Increment iteration number.
        return loss

In [3]:
weights = torch.nn.Parameter(5 * torch.randn((10, 10)))
opt = SGD([weights], lr=1)
for t in range(10):
    opt.zero_grad()  # Reset the gradients for all learnable parameters.
    loss = (weights**2).mean() # Compute a scalar loss value.
    print('step: ', t, ' loss: ', loss.cpu().item())
    loss.backward() # Run backward pass, which computes gradients.
    opt.step() # Run optimizer step.



step:  0  loss:  30.511552810668945
step:  1  loss:  29.303295135498047
step:  2  loss:  28.480331420898438
step:  3  loss:  27.826406478881836
step:  4  loss:  27.272661209106445
step:  5  loss:  26.786972045898438
step:  6  loss:  26.351327896118164
step:  7  loss:  25.954439163208008
step:  8  loss:  25.588685989379883
step:  9  loss:  25.248641967773438


In [4]:
weights = torch.nn.Parameter(5 * torch.randn((10, 10)))
opt = SGD([weights], lr=1e1)
for t in range(10):
    opt.zero_grad()  # Reset the gradients for all learnable parameters.
    loss = (weights**2).mean() # Compute a scalar loss value.
    print('step: ', t, ' loss: ', loss.cpu().item())
    loss.backward() # Run backward pass, which computes gradients.
    opt.step() # Run optimizer step.

step:  0  loss:  27.536653518676758
step:  1  loss:  17.623456954956055
step:  2  loss:  12.991262435913086
step:  3  loss:  10.164274215698242
step:  4  loss:  8.233061790466309
step:  5  loss:  6.826150894165039
step:  6  loss:  5.756953716278076
step:  7  loss:  4.919480800628662
step:  8  loss:  4.248358726501465
step:  9  loss:  3.7007925510406494


In [5]:
weights = torch.nn.Parameter(5 * torch.randn((10, 10)))
opt = SGD([weights], lr=1e2)
for t in range(10):
    opt.zero_grad()  # Reset the gradients for all learnable parameters.
    loss = (weights**2).mean() # Compute a scalar loss value.
    print('step: ', t, ' loss: ', loss.cpu().item())
    loss.backward() # Run backward pass, which computes gradients.
    opt.step() # Run optimizer step.

step:  0  loss:  22.864364624023438
step:  1  loss:  22.864364624023438
step:  2  loss:  3.9229040145874023
step:  3  loss:  0.0938839241862297
step:  4  loss:  1.879513302746987e-16
step:  5  loss:  2.09483407410993e-18
step:  6  loss:  7.05404375242678e-20
step:  7  loss:  4.202142943048303e-21
step:  8  loss:  3.6048691644032377e-22
step:  9  loss:  4.005410322912203e-23


In [6]:
weights = torch.nn.Parameter(5 * torch.randn((10, 10)))
opt = SGD([weights], lr=1e3)
for t in range(10):
    opt.zero_grad()  # Reset the gradients for all learnable parameters.
    loss = (weights**2).mean() # Compute a scalar loss value.
    print('step: ', t, ' loss: ', loss.cpu().item())
    loss.backward() # Run backward pass, which computes gradients.
    opt.step() # Run optimizer step.

step:  0  loss:  30.134939193725586
step:  1  loss:  10878.712890625
step:  2  loss:  1878924.75
step:  3  loss:  209010320.0
step:  4  loss:  16929836032.0
step:  5  loss:  1068466569216.0
step:  6  loss:  54851585179648.0
step:  7  loss:  2359949774553088.0
step:  8  loss:  8.698270847074304e+16
step:  9  loss:  2.7931116531498353e+18


when lr = 1: loss 下降速度缓慢
when lr = 1e1: loss下降速度加快
when lr = 1e2: loss下降速度最快,并且loss几乎下降到0
when lr = 1e3: loss随step迅速上升